In [ ]:
import io
import csv
import numpy as np
import pandas as pd

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

!pip install -q gradio
import gradio as gr

In [ ]:
uploaded = files.upload()  # opens a file picker; select your CSV/TXT dataset

uploaded_filename = list(uploaded.keys())[0]
raw_bytes = uploaded[uploaded_filename]
print(f"Loaded uploaded file: {uploaded_filename} ({len(raw_bytes)} bytes)")

Saving restaurant_recommendation_dataset.csv to restaurant_recommendation_dataset (1).csv
Loaded uploaded file: restaurant_recommendation_dataset (1).csv (8578098 bytes)


In [ ]:
# Base columns per the project spec
base_columns = [
    "Restaurant", "Food", "Category", "Price", "Rating", "Taste",
    "Quality", "Service", "ValueForMoney", "Distance", "Reviews",
    "Open24Hours", "Vegetarian", "LowSugar", "LowCalorie",
    "AllergyFriendly", "RecommendationScore"
]
columns_with_id = ["ID"] + base_columns

# --- Detect delimiter ---
sample_text = raw_bytes.decode("utf-8", errors="replace")
first_line = sample_text.splitlines()[0]

try:
    dialect = csv.Sniffer().sniff(first_line, delimiters=[",", "\t", ";"])
    detected_sep = dialect.delimiter
except csv.Error:
    detected_sep = "\t" if first_line.count("\t") >= first_line.count(",") else ","

print(f"Detected separator: {repr(detected_sep)}")

# --- Detect header row and whether ID column is present ---
first_line_fields = [f.strip() for f in first_line.split(detected_sep)]
field_count = len(first_line_fields)

lower_fields = [f.lower() for f in first_line_fields]
has_header_with_id = lower_fields == [c.lower() for c in columns_with_id]
has_header_no_id = lower_fields == [c.lower() for c in base_columns]

if has_header_with_id:
    columns = columns_with_id
    df = pd.read_csv(io.BytesIO(raw_bytes), sep=detected_sep, header=0)
    df.columns = columns
elif has_header_no_id:
    columns = base_columns
    df = pd.read_csv(io.BytesIO(raw_bytes), sep=detected_sep, header=0)
    df.columns = columns
else:
    columns = columns_with_id if field_count == len(columns_with_id) else base_columns
    df = pd.read_csv(io.BytesIO(raw_bytes), sep=detected_sep, header=None, names=columns)

print(f"Using columns: {columns}")
print(f"Loaded dataframe shape: {df.shape}")
df.head()

Detected separator: ','
Using columns: ['Restaurant', 'Food', 'Category', 'Price', 'Rating', 'Taste', 'Quality', 'Service', 'ValueForMoney', 'Distance', 'Reviews', 'Open24Hours', 'Vegetarian', 'LowSugar', 'LowCalorie', 'AllergyFriendly', 'RecommendationScore']
Loaded dataframe shape: (100000, 17)


,Restaurant,Food,Category,Price,Rating,Taste,Quality,Service,ValueForMoney,Distance,Reviews,Open24Hours,Vegetarian,LowSugar,LowCalorie,AllergyFriendly,RecommendationScore
0,Green Palace,Kebab,Mughlai,190,4.0,3.4,5.0,4.2,4.2,3.6,76,No,Yes,Yes,Yes,No,4.04
1,Flavor Point,Kulfi,Desserts,290,4.3,3.0,3.7,3.1,3.0,4.9,215,No,No,No,Yes,No,3.46
2,Metro Restaurant,Schezwan Rice,Chinese,270,3.5,3.4,3.3,2.2,2.1,2.4,370,No,Yes,No,Yes,No,2.91
3,Green Bites,Puff,Bakery,220,4.6,4.1,3.4,2.8,3.5,6.9,175,No,Yes,Yes,Yes,No,3.84
4,Tasty Eatery,Chaat,Street Food,190,4.1,3.6,4.6,4.8,4.0,1.4,262,No,No,Yes,No,No,4.26


In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows. Remaining rows: {after}")

print("\nMissing values per column:")
print(df.isnull().sum())

required_cols = [
    "Price", "Rating", "Taste", "Quality", "Service", "ValueForMoney",
    "Distance", "Reviews", "Open24Hours", "Vegetarian", "LowSugar",
    "LowCalorie", "AllergyFriendly", "Category", "RecommendationScore"
]
before_na = len(df)
df = df.dropna(subset=required_cols)
print(f"\nDropped {before_na - len(df)} rows with missing required values. Final rows: {len(df)}")

Removed 0 duplicate rows. Remaining rows: 100000

Missing values per column:
Restaurant             0
Food                   0
Category               0
Price                  0
Rating                 0
Taste                  0
Quality                0
Service                0
ValueForMoney          0
Distance               0
Reviews                0
Open24Hours            0
Vegetarian             0
LowSugar               0
LowCalorie             0
AllergyFriendly        0
RecommendationScore    0
dtype: int64

Dropped 0 rows with missing required values. Final rows: 100000


In [ ]:
numeric_features = ["Price", "Rating", "Taste", "Quality", "Service", "ValueForMoney", "Distance", "Reviews"]
categorical_features = ["Open24Hours", "Vegetarian", "LowSugar", "LowCalorie", "AllergyFriendly", "Category"]
feature_columns = numeric_features + categorical_features
target_column = "RecommendationScore"

X = df[feature_columns]
y = df[target_column]

print("Feature columns:", feature_columns)
print("Target column:", target_column)

Feature columns: ['Price', 'Rating', 'Taste', 'Quality', 'Service', 'ValueForMoney', 'Distance', 'Reviews', 'Open24Hours', 'Vegetarian', 'LowSugar', 'LowCalorie', 'AllergyFriendly', 'Category']
Target column: RecommendationScore


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 80000, Test size: 20000


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
model_pipeline.fit(X_train, y_train)
print("Model training complete.")

Model training complete.


In [ ]:
y_pred = model_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

MAE:  0.2102
RMSE: 0.2640
R²:   0.6023


In [ ]:
cat_choices = {
    col: sorted(df[col].dropna().unique().tolist(), key=str)
    for col in categorical_features
}

numeric_ranges = {
    col: (float(df[col].min()), float(df[col].max()), float(df[col].mean()))
    for col in numeric_features
}

print("Categorical choices:", cat_choices)
print("Numeric ranges (min, max, mean):", numeric_ranges)

Categorical choices: {'Open24Hours': ['No', 'Yes'], 'Vegetarian': ['No', 'Yes'], 'LowSugar': ['No', 'Yes'], 'LowCalorie': ['No', 'Yes'], 'AllergyFriendly': ['No', 'Yes'], 'Category': ['BBQ', 'Bakery', 'Beverages', 'Chinese', 'Continental', 'Desserts', 'Fast Food', 'Italian', 'Mexican', 'Mughlai', 'North Indian', 'Salads', 'Seafood', 'South Indian', 'Street Food', 'Thai']}
Numeric ranges (min, max, mean): {'Price': (50.0, 1480.0, 280.3663), 'Rating': (1.3, 5.0, 3.7958330000000005), 'Taste': (1.0, 5.0, 3.879361), 'Quality': (1.0, 5.0, 3.788815), 'Service': (1.0, 5.0, 3.6843220000000008), 'ValueForMoney': (1.0, 5.0, 3.5820769999999995), 'Distance': (0.1, 25.0, 3.523019), 'Reviews': (1.0, 14306.0, 185.98788)}


In [ ]:
def recommend_restaurants(Category, Vegetarian, LowSugar, LowCalorie, AllergyFriendly, Open24Hours,
                           min_rating, max_price, max_distance, top_n):
    filtered = df.copy()

    if Category != "Any":
        filtered = filtered[filtered["Category"] == Category]
    if Vegetarian != "Any":
        filtered = filtered[filtered["Vegetarian"] == Vegetarian]
    if LowSugar != "Any":
        filtered = filtered[filtered["LowSugar"] == LowSugar]
    if LowCalorie != "Any":
        filtered = filtered[filtered["LowCalorie"] == LowCalorie]
    if AllergyFriendly != "Any":
        filtered = filtered[filtered["AllergyFriendly"] == AllergyFriendly]
    if Open24Hours != "Any":
        filtered = filtered[filtered["Open24Hours"] == Open24Hours]

    filtered = filtered[
        (filtered["Rating"] >= min_rating) &
        (filtered["Price"] <= max_price) &
        (filtered["Distance"] <= max_distance)
    ]

    if filtered.empty:
        return pd.DataFrame([{"Message": "No restaurants match these filters. Try relaxing them."}])

    result = filtered.sort_values("RecommendationScore", ascending=False).head(int(top_n))
    result = result[[
        "Restaurant", "Food", "Category", "Price", "Rating", "Distance",
        "Vegetarian", "LowSugar", "LowCalorie", "AllergyFriendly",
        "Open24Hours", "RecommendationScore"
    ]]
    return result.reset_index(drop=True)

In [ ]:
def choices_with_any(col):
    return ["Any"] + sorted(df[col].dropna().unique().tolist(), key=str)

category_choices = choices_with_any("Category")
vegetarian_choices = choices_with_any("Vegetarian")
lowsugar_choices = choices_with_any("LowSugar")
lowcalorie_choices = choices_with_any("LowCalorie")
allergy_choices = choices_with_any("AllergyFriendly")
open24_choices = choices_with_any("Open24Hours")

price_min, price_max = float(df["Price"].min()), float(df["Price"].max())
distance_min, distance_max = float(df["Distance"].min()), float(df["Distance"].max())
rating_min, rating_max = float(df["Rating"].min()), float(df["Rating"].max())

custom_css = """
#header {text-align: center; padding: 10px 0 4px 0;}
#header h1 {font-size: 2.1em; margin-bottom: 4px;}
#find-btn {font-size: 1.05em; height: 48px;}
#summary-card {
    border-radius: 12px; padding: 16px 20px; margin-bottom: 10px;
    background: linear-gradient(135deg, #ff7a45 0%, #ff4d6d 100%);
    color: white;
}
#summary-card h3 {margin: 0 0 6px 0;}
.gr-box {border-radius: 12px !important;}
"""

def recommend_restaurants_ui(Category, Vegetarian, LowSugar, LowCalorie, AllergyFriendly, Open24Hours,
                              min_rating, max_price, max_distance, top_n):
    result = recommend_restaurants(Category, Vegetarian, LowSugar, LowCalorie, AllergyFriendly,
                                    Open24Hours, min_rating, max_price, max_distance, top_n)

    if "Message" in result.columns:
        summary = "⚠️ **No matches found** — try relaxing your filters."
        return result, summary

    top = result.iloc[0]
    summary = (
        f"### 🏆 Top Pick: {top['Restaurant']}\n"
        f"**{top['Food']}** · {top['Category']} · ⭐ {top['Rating']} · "
        f"💰 {top['Price']} · 📍 {top['Distance']} km · "
        f"Score: **{top['RecommendationScore']:.2f}**\n\n"
        f"Showing **{len(result)}** matching restaurant(s) below, ranked by RecommendationScore."
    )
    return result, summary

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange", secondary_hue="rose"), css=custom_css,
               title="Restaurant Recommendation System") as app:

    with gr.Column(elem_id="header"):
        gr.Markdown("# 🍽️ Restaurant Recommendation System")
        gr.Markdown("Find restaurants that match exactly what you're craving.")

    with gr.Row():
        with gr.Column(scale=1, min_width=320):
            with gr.Group():
                gr.Markdown("### 🍜 What are you in the mood for?")
                Category = gr.Dropdown(choices=category_choices, value="Any", label="Category")

                with gr.Row():
                    Vegetarian = gr.Dropdown(choices=vegetarian_choices, value="Any", label="Vegetarian")
                    Open24Hours = gr.Dropdown(choices=open24_choices, value="Any", label="Open 24 Hrs")

            with gr.Accordion("🥗 Dietary Preferences", open=False):
                LowSugar = gr.Dropdown(choices=lowsugar_choices, value="Any", label="Low Sugar")
                LowCalorie = gr.Dropdown(choices=lowcalorie_choices, value="Any", label="Low Calorie")
                AllergyFriendly = gr.Dropdown(choices=allergy_choices, value="Any", label="Allergy Friendly")

            with gr.Group():
                gr.Markdown("### 🎚️ Your Limits")
                min_rating = gr.Number(value=round(rating_min, 1), minimum=rating_min, maximum=rating_max,
                                        label=f"Minimum Rating ({rating_min}–{rating_max})")
                max_price = gr.Number(value=round(price_max, 0), minimum=price_min, maximum=price_max,
                                       label=f"Maximum Price ({int(price_min)}–{int(price_max)})")
                max_distance = gr.Number(value=round(distance_max, 1), minimum=distance_min, maximum=distance_max,
                                          label=f"Maximum Distance in km ({distance_min}–{distance_max})")
                top_n = gr.Slider(minimum=1, maximum=25, value=10, step=1, label="Number of Recommendations")

            submit_btn = gr.Button("🔍 Find My Restaurants", variant="primary", elem_id="find-btn")
            clear_btn = gr.Button("↺ Reset Filters", variant="secondary")

        with gr.Column(scale=2, min_width=420):
            summary_box = gr.Markdown(elem_id="summary-card", value="Set your filters and hit **Find My Restaurants**.")
            output_table = gr.Dataframe(label="📋 Ranked Results", wrap=True, interactive=False)

    submit_btn.click(
        fn=recommend_restaurants_ui,
        inputs=[Category, Vegetarian, LowSugar, LowCalorie, AllergyFriendly, Open24Hours,
                min_rating, max_price, max_distance, top_n],
        outputs=[output_table, summary_box]
    )

    clear_btn.click(
        fn=lambda: ("Any", "Any", "Any", "Any", "Any", "Any", round(rating_min, 1),
                     round(price_max, 0), round(distance_max, 1), 10, None,
                     "Set your filters and hit **Find My Restaurants**."),
        outputs=[Category, Vegetarian, LowSugar, LowCalorie, AllergyFriendly, Open24Hours,
                 min_rating, max_price, max_distance, top_n, output_table, summary_box]
    )

/tmp/ipykernel_708/1430456220.py:47: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange", secondary_hue="rose"), css=custom_css,


In [ ]:
# ============================================================
# GRADIO UI - RESTAURANT RECOMMENDATION SYSTEM
# ============================================================

import gradio as gr
import pandas as pd
import numpy as np


# ============================================================
# 1. RECOMMENDATION FUNCTION
# ============================================================

def recommend_restaurants(
    Category,
    Vegetarian,
    LowSugar,
    LowCalorie,
    AllergyFriendly,
    Open24Hours,
    min_rating,
    max_price,
    max_distance,
    top_n
):

    filtered = df.copy()

    if Category != "Any":
        filtered = filtered[filtered["Category"] == Category]

    if Vegetarian != "Any":
        filtered = filtered[filtered["Vegetarian"] == Vegetarian]

    if LowSugar != "Any":
        filtered = filtered[filtered["LowSugar"] == LowSugar]

    if LowCalorie != "Any":
        filtered = filtered[filtered["LowCalorie"] == LowCalorie]

    if AllergyFriendly != "Any":
        filtered = filtered[
            filtered["AllergyFriendly"] == AllergyFriendly
        ]

    if Open24Hours != "Any":
        filtered = filtered[
            filtered["Open24Hours"] == Open24Hours
        ]

    filtered = filtered[
        (filtered["Rating"] >= min_rating)
        & (filtered["Price"] <= max_price)
        & (filtered["Distance"] <= max_distance)
    ]

    if filtered.empty:

        return pd.DataFrame([
            {
                "Message":
                "No restaurants match these filters."
            }
        ])

    result = (
        filtered
        .sort_values(
            "RecommendationScore",
            ascending=False
        )
        .head(int(top_n))
    )

    result = result[
        [
            "Restaurant",
            "Food",
            "Category",
            "Price",
            "Rating",
            "Distance",
            "Vegetarian",
            "LowSugar",
            "LowCalorie",
            "AllergyFriendly",
            "Open24Hours",
            "RecommendationScore"
        ]
    ]

    return result.reset_index(drop=True)


# ============================================================
# 2. DROPDOWN OPTIONS
# ============================================================

def choices_with_any(col):

    return (
        ["Any"]
        + sorted(
            df[col].dropna().unique().tolist(),
            key=str
        )
    )


category_choices = choices_with_any("Category")
vegetarian_choices = choices_with_any("Vegetarian")
lowsugar_choices = choices_with_any("LowSugar")
lowcalorie_choices = choices_with_any("LowCalorie")
allergy_choices = choices_with_any("AllergyFriendly")
open24_choices = choices_with_any("Open24Hours")


# ============================================================
# 3. DATA RANGES
# ============================================================

price_min = float(df["Price"].min())
price_max = float(df["Price"].max())

distance_min = float(df["Distance"].min())
distance_max = float(df["Distance"].max())

rating_min = float(df["Rating"].min())
rating_max = float(df["Rating"].max())


# ============================================================
# 4. RESULT DISPLAY FUNCTION
# ============================================================

def recommend_restaurants_ui(
    Category,
    Vegetarian,
    LowSugar,
    LowCalorie,
    AllergyFriendly,
    Open24Hours,
    min_rating,
    max_price,
    max_distance,
    top_n
):

    result = recommend_restaurants(
        Category,
        Vegetarian,
        LowSugar,
        LowCalorie,
        AllergyFriendly,
        Open24Hours,
        min_rating,
        max_price,
        max_distance,
        top_n
    )

    # --------------------------------------------------------
    # NO RESULTS
    # --------------------------------------------------------

    if "Message" in result.columns:

        summary = """
        <div class="empty-card">

            <div class="empty-icon">🍽️</div>

            <h2>No matching restaurants</h2>

            <p>
                We couldn't find a restaurant matching
                all your selected preferences.
            </p>

            <p>
                Try relaxing your rating, price,
                distance or dietary filters.
            </p>

        </div>
        """

        return result, summary


    # --------------------------------------------------------
    # TOP RESTAURANT
    # --------------------------------------------------------

    top = result.iloc[0]

    restaurant_name = str(top["Restaurant"])
    food = str(top["Food"])
    category = str(top["Category"])

    rating = top["Rating"]
    price = top["Price"]
    distance = top["Distance"]
    score = top["RecommendationScore"]

    summary = f"""

    <div class="recommendation-card">

        <div class="recommendation-badge">
            ⭐ &nbsp; TOP RECOMMENDATION
        </div>

        <div class="restaurant-name">
            {restaurant_name}
        </div>

        <div class="restaurant-food">
            🍢 &nbsp; {food} &nbsp; • &nbsp; {category}
        </div>


        <div class="stats-row">

            <div class="stat-card rating-card">

                <div class="stat-icon">
                    ⭐
                </div>

                <div class="stat-content">

                    <div class="stat-label">
                        Rating
                    </div>

                    <div class="stat-value rating-value">
                        {rating:.1f}
                    </div>

                </div>

            </div>


            <div class="stat-card price-card">

                <div class="stat-icon">
                    💰
                </div>

                <div class="stat-content">

                    <div class="stat-label">
                        Price
                    </div>

                    <div class="stat-value price-value">
                        {price:.0f}
                    </div>

                </div>

            </div>


            <div class="stat-card distance-card">

                <div class="stat-icon">
                    📍
                </div>

                <div class="stat-content">

                    <div class="stat-label">
                        Distance
                    </div>

                    <div class="stat-value distance-value">
                        {distance:.1f} km
                    </div>

                </div>

            </div>


            <div class="stat-card score-card">

                <div class="stat-icon">
                    🎯
                </div>

                <div class="stat-content">

                    <div class="stat-label">
                        Score
                    </div>

                    <div class="stat-value score-value">
                        {score:.2f}
                    </div>

                </div>

            </div>

        </div>


        <div class="showing-text">

            ✨ &nbsp;
            Showing <b>{len(result)}</b>
            restaurant(s), ranked by recommendation score.

        </div>

    </div>
    """

    return result, summary


# ============================================================
# 5. RESET FUNCTION
# ============================================================

def reset_filters():

    return (

        "Any",
        "Any",
        "Any",
        "Any",
        "Any",
        "Any",

        round(rating_min, 1),
        round(price_max, 0),
        round(distance_max, 1),

        10,

        None,

        """
        <div class="welcome-card">

            <div class="welcome-icon">
                🍽️
            </div>

            <h2>
                Find Your Perfect Restaurant
            </h2>

            <p>
                Select your preferences and click
                <b>Find My Restaurants</b>.
            </p>

        </div>
        """
    )


# ============================================================
# 6. CUSTOM CSS
# ============================================================

custom_css = """

/* ==========================================================
   MAIN BACKGROUND
   ========================================================== */

body {
    background: #020617 !important;
}

.gradio-container {
    background: #020617 !important;
    max-width: 1450px !important;
}


/* ==========================================================
   HERO
   ========================================================== */

.hero {
    padding: 38px 40px;
    margin-bottom: 25px;

    border-radius: 22px;

    background:
        linear-gradient(
            120deg,
            #ef4444 0%,
            #ec4899 45%,
            #f97316 100%
        );

    color: white;

    box-shadow:
        0 12px 35px rgba(0,0,0,0.35);
}

.hero-title {
    font-size: 42px;
    font-weight: 800;
    margin-bottom: 8px;
}

.hero-subtitle {
    font-size: 18px;
    opacity: 0.95;
}


/* ==========================================================
   FILTER PANEL
   ========================================================== */

.filter-panel {
    background: #0f172a !important;

    border: 1px solid #1e293b !important;

    border-radius: 18px !important;

    padding: 8px !important;
}


/* ==========================================================
   HEADINGS
   ========================================================== */

.section-heading {
    color: white !important;
    font-size: 20px !important;
    font-weight: 700 !important;
}


/* ==========================================================
   INPUT LABELS
   ========================================================== */

label {
    color: #e2e8f0 !important;
}


/* ==========================================================
   DROPDOWNS / INPUTS
   ========================================================== */

.gradio-container input,
.gradio-container textarea {

    background: #111827 !important;

    color: #f8fafc !important;

    border-color: #334155 !important;
}


/* ==========================================================
   TOP RECOMMENDATION CARD
   ========================================================== */

.recommendation-card {

    padding: 38px;

    border-radius: 25px;

    margin-bottom: 22px;

    color: white;

    background:
        linear-gradient(
            135deg,
            #7c3aed 0%,
            #2563eb 30%,
            #0891b2 52%,
            #10b981 73%,
            #f59e0b 100%
        );

    box-shadow:
        0 18px 45px rgba(0,0,0,0.35);

    border:
        1px solid rgba(255,255,255,0.35);
}


/* ==========================================================
   BADGE
   ========================================================== */

.recommendation-badge {

    display: inline-block;

    padding: 10px 18px;

    border-radius: 10px;

    background:
        rgba(54, 20, 100, 0.80);

    color: white;

    font-size: 14px;

    font-weight: 800;

    letter-spacing: 1px;

    margin-bottom: 20px;
}


/* ==========================================================
   RESTAURANT NAME
   ========================================================== */

.restaurant-name {

    font-size: 43px;

    font-weight: 850;

    color: white;

    margin-bottom: 10px;

    text-shadow:
        0 3px 12px rgba(0,0,0,0.25);
}


/* ==========================================================
   FOOD / CATEGORY
   ========================================================== */

.restaurant-food {

    display: inline-block;

    background: rgba(255,255,255,0.92);

    color: #312e81;

    padding: 11px 20px;

    border-radius: 12px;

    font-size: 17px;

    font-weight: 700;

    margin-bottom: 28px;
}


/* ==========================================================
   STATS ROW
   ========================================================== */

.stats-row {

    display: flex;

    gap: 18px;

    flex-wrap: wrap;
}


/* ==========================================================
   STAT CARD
   ========================================================== */

.stat-card {

    flex: 1;

    min-width: 170px;

    display: flex;

    align-items: center;

    gap: 14px;

    padding: 18px;

    border-radius: 18px;

    background:
        rgba(2, 6, 23, 0.45);

    border:
        1px solid rgba(255,255,255,0.35);

    backdrop-filter: blur(8px);

    box-shadow:
        inset 0 0 15px rgba(255,255,255,0.04);
}


/* ==========================================================
   ICON
   ========================================================== */

.stat-icon {

    width: 58px;

    height: 58px;

    border-radius: 50%;

    display: flex;

    align-items: center;

    justify-content: center;

    font-size: 27px;

    background:
        rgba(2,6,23,0.45);

    border:
        1px solid rgba(255,255,255,0.55);
}


/* ==========================================================
   STAT LABEL
   ========================================================== */

.stat-label {

    font-size: 14px;

    font-weight: 600;

    color: #f8fafc;

    opacity: 0.9;
}


/* ==========================================================
   STAT VALUES
   ========================================================== */

.stat-value {

    font-size: 29px;

    font-weight: 850;

    margin-top: 2px;
}


/* ==========================================================
   COLOUR WHEEL BASED ACCENTS
   ========================================================== */

/* Yellow */

.rating-value {
    color: #facc15;
}


/* Cyan / Turquoise */

.price-value {
    color: #2dd4bf;
}


/* Purple */

.distance-value {
    color: #c084fc;
}


/* Pink / Magenta */

.score-value {
    color: #fb7185;
}


/* ==========================================================
   SHOWING TEXT
   ========================================================== */

.showing-text {

    display: inline-block;

    margin-top: 28px;

    padding: 12px 18px;

    border-radius: 10px;

    background:
        rgba(2,6,23,0.45);

    color: #f8fafc;

    font-size: 15px;
}


/* ==========================================================
   EMPTY STATE
   ========================================================== */

.empty-card {

    padding: 40px;

    border-radius: 20px;

    text-align: center;

    background: #0f172a;

    border:
        1px solid #334155;

    color: #f8fafc;
}

.empty-icon {

    font-size: 45px;

    margin-bottom: 10px;
}


/* ==========================================================
   WELCOME STATE
   ========================================================== */

.welcome-card {

    padding: 45px;

    border-radius: 20px;

    text-align: center;

    background:
        linear-gradient(
            135deg,
            #111827,
            #1e1b4b
        );

    border:
        1px solid #334155;

    color: white;
}

.welcome-icon {

    font-size: 50px;

    margin-bottom: 10px;
}

.welcome-card h2 {

    color: white;

    font-size: 25px;
}

.welcome-card p {

    color: #cbd5e1;

    font-size: 16px;
}


/* ==========================================================
   SEARCH BUTTON
   ========================================================== */

#find-btn {

    height: 58px !important;

    border-radius: 14px !important;

    font-size: 17px !important;

    font-weight: 800 !important;

    background:
        linear-gradient(
            90deg,
            #f97316,
            #ec4899
        ) !important;

    border: none !important;

    color: white !important;

    box-shadow:
        0 8px 25px rgba(236,72,153,0.30) !important;

    transition: 0.2s ease !important;
}

#find-btn:hover {

    transform: translateY(-2px);

    box-shadow:
        0 12px 30px rgba(236,72,153,0.45) !important;
}


/* ==========================================================
   RESET BUTTON
   ========================================================== */

#reset-btn {

    height: 48px !important;

    border-radius: 14px !important;

    background: #1e293b !important;

    border: 1px solid #475569 !important;

    color: #e2e8f0 !important;
}


/* ==========================================================
   DATAFRAME
   ========================================================== */

.results-table {

    border-radius: 16px !important;

    overflow: hidden !important;
}


/* ==========================================================
   TABLE TEXT
   ========================================================== */

.results-table {

    color: #f8fafc !important;
}


/* ==========================================================
   RESPONSIVE
   ========================================================== */

@media (max-width: 900px) {

    .restaurant-name {
        font-size: 32px;
    }

    .stat-card {
        min-width: 140px;
    }

}


@media (max-width: 600px) {

    .hero-title {
        font-size: 30px;
    }

    .restaurant-name {
        font-size: 28px;
    }

    .recommendation-card {
        padding: 25px;
    }

    .stats-row {
        flex-direction: column;
    }

    .stat-card {
        width: 100%;
    }

}
"""


# ============================================================
# 7. BUILD GRADIO APP
# ============================================================

with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue="orange",
        secondary_hue="pink",
        neutral_hue="slate"
    ),
    css=custom_css,
    title="Restaurant Recommendation System"
) as app:


    # ========================================================
    # HERO HEADER
    # ========================================================

    gr.HTML(
        """
        <div class="hero">

            <div class="hero-title">
                🍽️ Restaurant Recommendation System
            </div>

            <div class="hero-subtitle">
                Find restaurants that match your taste,
                dietary preferences, budget and distance.
            </div>

        </div>
        """
    )


    # ========================================================
    # MAIN LAYOUT
    # ========================================================

    with gr.Row(equal_height=False):


        # ====================================================
        # LEFT FILTER SECTION
        # ====================================================

        with gr.Column(
            scale=1,
            min_width=320
        ):

            with gr.Group(
                elem_classes="filter-panel"
            ):

                gr.Markdown(
                    "### 🍜 What are you in the mood for?",
                    elem_classes="section-heading"
                )


                Category = gr.Dropdown(
                    choices=category_choices,
                    value="Any",
                    label="Category"
                )


                with gr.Row():

                    Vegetarian = gr.Dropdown(
                        choices=vegetarian_choices,
                        value="Any",
                        label="🌱 Vegetarian"
                    )

                    Open24Hours = gr.Dropdown(
                        choices=open24_choices,
                        value="Any",
                        label="🌙 Open 24 Hrs"
                    )


            # =================================================
            # DIETARY PREFERENCES
            # =================================================

            with gr.Accordion(
                "🥗 Dietary Preferences",
                open=True
            ):

                LowSugar = gr.Dropdown(
                    choices=lowsugar_choices,
                    value="Any",
                    label="Low Sugar"
                )

                LowCalorie = gr.Dropdown(
                    choices=lowcalorie_choices,
                    value="Any",
                    label="Low Calorie"
                )

                AllergyFriendly = gr.Dropdown(
                    choices=allergy_choices,
                    value="Any",
                    label="Allergy Friendly"
                )


            # =================================================
            # LIMITS
            # =================================================

            with gr.Group(
                elem_classes="filter-panel"
            ):

                gr.Markdown(
                    "### 🎚️ Your Limits",
                    elem_classes="section-heading"
                )


                min_rating = gr.Slider(
                    minimum=rating_min,
                    maximum=rating_max,
                    value=round(rating_min, 1),
                    step=0.1,
                    label="⭐ Minimum Rating"
                )


                max_price = gr.Slider(
                    minimum=price_min,
                    maximum=price_max,
                    value=round(price_max, 0),
                    step=1,
                    label="💰 Maximum Price"
                )


                max_distance = gr.Slider(
                    minimum=distance_min,
                    maximum=distance_max,
                    value=round(distance_max, 1),
                    step=0.1,
                    label="📍 Maximum Distance (km)"
                )


                top_n = gr.Slider(
                    minimum=1,
                    maximum=25,
                    value=10,
                    step=1,
                    label="🏆 Number of Recommendations"
                )


            # =================================================
            # BUTTONS
            # =================================================

            submit_btn = gr.Button(
                "🔍 Find My Restaurants",
                variant="primary",
                elem_id="find-btn"
            )


            clear_btn = gr.Button(
                "↺ Reset Filters",
                variant="secondary",
                elem_id="reset-btn"
            )


        # ====================================================
        # RIGHT RESULTS SECTION
        # ====================================================

        with gr.Column(
            scale=2,
            min_width=500
        ):


            # =================================================
            # TOP RECOMMENDATION
            # =================================================

            summary_box = gr.HTML(
                """
                <div class="welcome-card">

                    <div class="welcome-icon">
                        🍽️
                    </div>

                    <h2>
                        Find Your Perfect Restaurant
                    </h2>

                    <p>
                        Set your preferences on the left and
                        click <b>Find My Restaurants</b>.
                    </p>

                </div>
                """
            )


            # =================================================
            # RESULTS TABLE
            # =================================================

            gr.Markdown(
                "### 📋 Ranked Restaurant Recommendations",
                elem_classes="section-heading"
            )


            output_table = gr.Dataframe(
                label=None,
                wrap=True,
                interactive=False,
                elem_classes="results-table"
            )


    # ========================================================
    # SEARCH EVENT
    # ========================================================

    submit_btn.click(
        fn=recommend_restaurants_ui,

        inputs=[
            Category,
            Vegetarian,
            LowSugar,
            LowCalorie,
            AllergyFriendly,
            Open24Hours,
            min_rating,
            max_price,
            max_distance,
            top_n
        ],

        outputs=[
            output_table,
            summary_box
        ]
    )


    # ========================================================
    # RESET EVENT
    # ========================================================

    clear_btn.click(
        fn=reset_filters,

        outputs=[
            Category,
            Vegetarian,
            LowSugar,
            LowCalorie,
            AllergyFriendly,
            Open24Hours,
            min_rating,
            max_price,
            max_distance,
            top_n,
            output_table,
            summary_box
        ]
    )


# ============================================================
# 8. LAUNCH
# ============================================================

app.launch()

/tmp/ipykernel_708/1731435563.py:939: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d2cc672058522169d1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
